### AITW-darcy_infiltration

Simulation of monomer infiltration into delignified wood templates, relevant to the fabrication of transparent wood composites. Full saturation of the porous wood by the monomer precursor is a critical step that governs both the optical transparency and mechanical integrity of the final material.

#### Links

- <a href="https://github.com/AI-TranspWood/AITW-Darcy-Infiltration-Model" target="_blank" rel="noopener noreferrer">Source code</a>
- <a href="https://www.eessi.io/docs/available_software/detail/AITW-darcy_infiltration" target="_blank" rel="noopener noreferrer">EESSI</a>

In [ ]:
from utilities import initialize_with_module

await initialize_with_module([
        'AITW-darcy_infiltration/1.0.0-foss-2023b',
        'matplotlib/3.8.2-gfbf-2023b',
        'h5py/3.11.0-foss-2023b',
    ],
    '2023.06'
)

In [ ]:
import os
import json
from io import BytesIO

import h5py
import numpy as np
from IPython.display import Image as IPyImage
import matplotlib.pyplot as plt

def output_finder(basedir: str) -> tuple[str, str]:
    """
    Find the output files in the given base directory.

    Parameters:
    - basedir: The base directory to search for output files.

    Returns:
    A tuple containing the paths to the output image and CSV files.
    """
    res = []
    for root, dirs, files in os.walk(basedir):
        for file in files:
            if file.endswith('.json'):
                basename = os.path.splitext(file)[0]
                if not os.path.exists(os.path.join(root, f"{basename}.log")):
                    continue
                if not os.path.exists(os.path.join(root, f"{basename}_solution.npy")):
                    continue
                res.append((basename, os.path.join(root, file)))
                
    return res

def output_loader(filepath: str) -> dict:
    """
    Load the output data from the given JSON file.

    Parameters:
    - filepath: The path to the JSON file containing the output data.

    Returns:
    A dictionary containing the loaded output data.
    """
    dirpath = os.path.dirname(filepath)
    basename = os.path.splitext(os.path.basename(filepath))[0]
    with open(filepath, 'r') as f:
        item = json.load(f)

    res = {'data': item}

    # H5 files orginized as:
    # - Mesh
    #   - mesh
    #     - geometry DATASET (NPTS, 3)
    #     - topology DATASET
    # - Function
    #   - S/u
    #     - <timestep> DATASET
    def _read_h5_file(h5f):
        data_dict = {}
        for key in h5f.keys():
            item = h5f[key]
            if isinstance(item, h5py.Group):
                data_dict[key] = _read_h5_file(item)
            elif isinstance(item, h5py.Dataset):
                data_dict[key] = item[()]
            else:
                print(f"Unknown data type: {type(item)}")
        return data_dict

    def read_h5_file(h5_filepath):
        with h5py.File(h5_filepath, 'r') as h5f:
            return _read_h5_file(h5f)


    if os.path.exists(os.path.join(dirpath, f"{basename}_output_S.h5")):
        data_dict_S = read_h5_file(os.path.join(dirpath, f"{basename}_output_S.h5"))
        res['S'] = data_dict_S
    if os.path.exists(os.path.join(dirpath, f"{basename}_output_u.h5")):
        data_dict_u = read_h5_file(os.path.join(dirpath, f"{basename}_output_u.h5"))
        res['u'] = data_dict_u

    return res

def output_visualizer(data: dict):
    """
    Visualize the output data using interactive sliders.

    Parameters:
    - data: A pandas DataFrame containing the output data to visualize.
    """
    dct = data.get('data', {})
    sat_time = dct.get('sat_time', None)
    input_params = dct.get('input_parameters', {})
    print(json.dumps(input_params, indent=4))
    if sat_time is not None:
        print('*' * 40)
        print(f'Saturation Time: {sat_time:.4f}')
        print('*' * 40)
    else:
        print('Saturation Time: Not available')

    output_box = wdg.Output()

    def display_function(name):
        timesteps = sorted(data[name]['Function'][name].keys())
        if not timesteps:
            print(f"No timesteps found for {name}.")
            return

        func_data = np.array([data[name]['Function'][name][t][:, 0] for t in timesteps])
        # print(f"Loaded {len(timesteps)} timesteps for {name}  func_data.shape {func_data.shape}.")
        # print(f"func min: {func_data.min()}, max: {func_data.max()}")

        dmin = np.round(func_data.min(), 2)
        dmax = np.round(func_data.max(), 2)

        timesteps_flt = [float(t.replace('_', '.')) for t in timesteps]
        timesteps_flt_sorted = sorted(timesteps_flt)
        timesteps_str = [f"{t:.4f}" for t in timesteps_flt]
        timesteps_str_sorted = [f"{t:.4f}" for t in timesteps_flt_sorted]
        # timestep_map = dict(zip(timesteps_str, timesteps))

        geometry = data[name]['Mesh']['mesh']['geometry']
        xvals = geometry[:, 0]
        yvals = geometry[:, 1]
        zvals = geometry[:, 2]

        X_unq = np.unique(xvals)
        Y_unq = np.unique(yvals)
        Z_unq = np.unique(zvals)

        ix = np.searchsorted(X_unq, xvals)
        iy = np.searchsorted(Y_unq, yvals)
        iz = np.searchsorted(Z_unq, zvals)

        X, Y, _ = np.meshgrid(X_unq, Y_unq, Z_unq, indexing='ij')

        shape = (len(X_unq), len(Y_unq), len(Z_unq))

        zvals_str = [f"{z:.4f}" for z in Z_unq]

        slider_z = wdg.SelectionSlider(
            value=zvals_str[len(zvals_str) // 2],
            options=zvals_str,
            description=f'Z value:',
            continuous_update=False,
            # layout=wdg.Layout(width='80%')
        )
        slider_time = wdg.SelectionSlider(
            value=timesteps_str_sorted[0],
            options=timesteps_str_sorted,
            description=f'{name} timestep:',
            continuous_update=False,
            layout=wdg.Layout(width='60%')
        )

        output_box_inner = wdg.Output()

        def update_plot(*args, **kwargs):
            timestep_str = slider_time.value
            t_idx = timesteps_str.index(timestep_str)
            # timestep = timestep_map[timestep_str]
            values = np.empty(shape)
            values[ix, iy, iz] = func_data[t_idx]
            values[np.isnan(values)] = dmax
            z_value = slider_z.value
            z_idx = zvals_str.index(z_value)

            # Print the section of the values at the middle Z index
            # mid_z_index = len(Z_unq) // 2

            fig, ax = plt.subplots(figsize=(6, 3.5))
            levels = np.linspace(dmin, dmax, 51)
            cf = ax.contourf(
                X[:, :, z_idx],
                Y[:, :, z_idx],
                values[:, :, z_idx],
                levels=levels,
                cmap='viridis',
                # extend="both",
                vmin=dmin, vmax=dmax,
            )
            cb = fig.colorbar(cf, ax=ax)
            ax.set_title(f'{name} at timestep {timestep_str}')
            ax.set_xlabel('X')
            ax.set_ylabel('Y')
            cf.set_label('Values')
            # Ensure CB also goes from 0 to 1
            cf.set_clim(dmin, dmax)
            cb.ax.set_ylim(dmin, dmax)
            cb.ax.set_yticks(np.linspace(dmin, dmax, 6))

            image_buffer = BytesIO()
            plt.tight_layout()
            plt.savefig(image_buffer, format='png', dpi=200)
            image_buffer.seek(0)
            image = IPyImage(data=image_buffer.read(), format='png')
            with output_box_inner:
                output_box_inner.clear_output(wait=True)
                display(image)

        slider_time.observe(update_plot, names='value')
        slider_z.observe(update_plot, names='value')

        start_interval = 300
        interval_widget = wdg.IntText(
            value=start_interval,
            description='Interval (ms):',
            disabled=False,
        )

        play_button = wdg.Play(
            value=0,
            min=0,
            max=len(timesteps_str) - 1,
            step=1,
            interval=start_interval,
            description="Play",
            disabled=False
        )
        wdg.jslink((play_button, 'value'), (slider_time, 'index'))
        wdg.jslink((interval_widget, 'value'), (play_button, 'interval'))

        time_hbox = wdg.HBox([slider_time, play_button, interval_widget])

        main_box = wdg.VBox([slider_z, time_hbox, output_box_inner])
        with output_box:
            output_box.clear_output(wait=True)
            display(main_box)

        update_plot()

    values = []

    if 'S' in data:
        values.append('S')
    if 'u' in data:
        values.append('u')

    value_selector = wdg.Dropdown(
        options=values,
        value=values[0] if values else None,
        description='Select Output:',
        disabled=False,
    )

    value_selector.observe(lambda change: display_function(change['new']), names='value')
    if value_selector.value is not None:
        display_function(value_selector.value)

    main_box = wdg.VBox([value_selector, output_box])
    display(main_box)



In [ ]:
import utilities
import ipywidgets as wdg
from common import DARCY_DEFAULTS, DARCY_OUTDIR
from aitw_darcy.cli import infiltration


widget_darcy, darcy_event = utilities.click_cli_widget(
    infiltration,
    title='Darcy Infiltration',
    description_text="",
    override_defaults=DARCY_DEFAULTS,
    visible_panel_names=['IO'],
    custom_names={'visualize': 'Write Paraview Files'},
    button_description='Run Darcy Infiltration',
    button_running_text='Running...',
)

def selector_label_func(label, data):
    data = data['data']
    input_params = data.get('input_parameters', {})
    dx = input_params['xmax'] - input_params['xmin']
    dy = input_params['ymax'] - input_params['ymin']
    dz = input_params['zmax'] - input_params['zmin']
    nx = input_params['nx']
    ny = input_params['ny']
    nz = input_params['nz']
    porosity = input_params['porosity']
    return f"{label}: ({nx}x{ny}x{nz}) in ({dx:.3f}x{dy:.3f}x{dz:.3f}), porosity={porosity:.3f}"

visualizer_accordion = utilities.output_visualizer_widget(
    title='Darcy Infiltration Results',
    file_format='json',
    event_observer=darcy_event,
    output_dir=DARCY_OUTDIR,
    outputs_func=lambda: output_finder(DARCY_OUTDIR),
    read_func=output_loader,
    data_display_func=output_visualizer,
    selector_label_func=selector_label_func,
    # data_display_func=data_display_func,
)
main_box = wdg.VBox(children=[widget_darcy, visualizer_accordion])
display(main_box)

In [ ]:
import utilities
utilities.final_spacer()